In [0]:
import streamlit as st
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from PyPDFLoader import PyPDFLoader

# Load environment variables
load_dotenv()

# Set up Azure Text Analytics client
azure_key = os.getenv("AZURE_TEXT_ANALYTICS_KEY")
azure_endpoint = os.getenv("AZURE_TEXT_ANALYTICS_ENDPOINT")
azure_credentials = AzureKeyCredential(azure_key)
text_analytics_client = TextAnalyticsClient(endpoint=azure_endpoint, credential=azure_credentials)

# Load content of the PDF file
pdf_loader = PyPDFLoader("JDB-Template.pdf")
pdf_text = pdf_loader.extract_text()

# Process user input and generate response
def generate_response(user_input):
    # Use Azure Text Analytics for natural language understanding
    response = text_analytics_client.analyze_sentiment([user_input])[0]
    sentiment_score = response.confidence_scores.positive
    if sentiment_score > 0.5:
        reply = "That sounds positive!"
    elif sentiment_score < 0.5:
        reply = "That sounds negative."
    else:
        reply = "I'm not sure how to interpret that."
        
    return reply

# Streamlit application
def main():
    st.title("PDF Document Q&A System")

    user_question = st.text_input("Please enter your question:")

    if st.button("Get Answer"):
        if user_question:
            answer = generate_response(user_question)
            st.write("Answer:", answer)
        else:
            st.warning("Please enter a question!")

if __name__ == "__main__":
    main()